In [ ]:
from IPython.display import display, HTML

display(HTML(
    """
    <head> 
        <style>
            p {text-align: left; font-family: Times New Roman; font-size: 12pt;}
            h1 {font-family: Times New Roman; font-size: 50pt; text-align: center}
        </style>
        <body>
            <h1>PROYECTO:<br>CURVA DE OFERTA AGREGADA</h1>
            <h3 style = "font-family: Times New Roman; text-align: center;">Juan Esteban Roa<br>Danna Alejandra Salguero</h3>
            <p style = "background-color: tomato">Este proyecto trata sobre la obtención del precio de bolsa a partir de la demanda pronosticada y la curva de oferta agregada, identificando agentes y tecnología que ponen el precio marginal.</p>
            <h2 style = "text-align:left; font-family:Times New Roman;">Conceptos:</h2>
            <p><b>Oferta agregada (mérito de despacho)</b>:Es la construcción horaria donde se ordenan todas las ofertas de energía de menor a mayor precio. El punto exacto en el que esta curva escalonada corta la demanda real de esa hora determina el Precio de Bolsa.</p>
            <p><b>Disponibilidad real</b>: La cantidad máxima de megavatios (MW) que la planta puede generar físicamente (limitada por agua en embalses, fallas mecánicas o combustible disponible).</p>
            <p><b>Precio de oferta</b>: El valor (en COP/kWh) al que está dispuesta a vender esa energía.</p>

            <h2 style = "text-align:left; font-family:Times New Roman;">Estrategia:</h2>
            <ol style = "font-family:Times New Roman">
                <li>Obtención de datos por medio de la API de XM.</li>
                <li>Delimitación del periodo</li>
                <li>Organización y tratamiento de los Dataframes</li>
                <li>Graficación de los datos</li>
            </ol>

        </body>
    </head>
    """
))

In [ ]:
from pydataxm.pydataxm import ReadDB
import datetime as dt    
import pandas as pd
import plotly.express as px

In [ ]:
display(HTML(
    """
    <head>
        <style>
            li {font-family: Times New Roman;}
            h1 {font-size: 30pt; font-family: Times New Roman;text-align: left}
        </style>
    </head>
    <body>
        <h1>1. Obtención de datos</h1>
        <h2>Fecha: 1 Octubre 2025 - 1 Diciembre 2025</h2>
        <ul> Se usarán las variables de:
            <li>Precio ofertado por los generadores</li>
            <li>Disponibilidad declarada por los generadores</li>
            <li>Precio de bolsa nacional</li>
            <li>Generación ideal</li>    
            <li>Proyección de demanda de UPME en escenario bajo</li>
        </ul>
        <h2 style = "text-align:left;">Detalles de las variables</h2>
        <img src = "disponibilidad.png"></img>
        <img src = "oferta.png"></img>
        <img src = "operadores.png"></img>
        <img src = "bolsa.png"></img>
        <h2 style = "color:#EDC721;">Demanda de Energia Proyectada UPME Escenario Bajo por Sistema:</h2>
        <p>Id = EscDemUPMEBajo. Type = MonthlyEntities. Units = kWh  </p>
    </body>
    """
))

In [ ]:
FECHA_INICIO = "2025-10-01"
FECHA_FIN = "2025-12-01"
METRICAS = {
    "oferta_precio": "PrecOferDesp",     # Precio de Oferta del Despacho   (Recurso)
    "disponibilidad": "DispoDeclarada",   # Disponibilidad Declarada        (Recurso)
    "gen_ideal": "GeneIdea",             # Generación Ideal por Recurso    (Recurso)
    "mpo": "MaxPrecOferNal",             # Máximo Precio de Oferta         (Sistema)
    "pbn": "PrecBolsNaci",               # Precio de Bolsa Nacional        (Sistema)
    "listRecursos":"ListadoRecursos",    #Listado de Recursos
    "listAgentes":"ListadoAgentes",       #Listado de Agentes
    "demanda_comercial":"DemaCome"

}

entidad = ["Recurso","Recurso","Recurso","Sistema","Sistema","Sistema","Sistema","Sistema"]

api = ReadDB()

df_variables = {}
i = 0
for name, code in METRICAS.items():
    var = api.request_data(code,entidad[i],FECHA_INICIO,FECHA_FIN)
    df_variables[name] = var
    i+=1

In [ ]:
agentes_nombres = (
    df_variables["listRecursos"].drop_duplicates(subset=['Values_Code'])
      .set_index('Values_Code')['Values_Name']
      .to_dict()
)

agentes_actividad = (df_variables["listRecursos"].drop_duplicates(subset = ['Values_Name']).set_index('Values_Name')['Values_Type'].to_dict())

In [ ]:
#nuevos_nombres_oferta = [f"oferta_hora_{x}" for x in range(1,25)]

df_oferta = df_variables["oferta_precio"]
df_oferta.rename(columns = {df_oferta.columns.values[1+i]:f"oferta_hora_{i}" for i in range(1,25) }, inplace = True)

#nuevos_nombres_dip = [f"disp_hora_{x}" for x in range(1,25)]

df_disponibilidad = df_variables["disponibilidad"]
df_disponibilidad.rename(columns = {df_disponibilidad.columns.values[1+i]:f"disp_hora_{i}" for i in range(1,25) }, inplace = True)


df_oferta['Date'] = pd.to_datetime(df_oferta['Date'])
df_disponibilidad['Date'] = pd.to_datetime(df_disponibilidad['Date'])

print(df_disponibilidad.head())

        Id Values_code  disp_hora_1  disp_hora_2  disp_hora_3  disp_hora_4  \
0  Recurso        2QBW        18420        18420        18420        18420   
1  Recurso        2QEK        35000        35000        35000        35000   
2  Recurso        2QRL          650          650          650          650   
3  Recurso        2QV2         2000         2000         2000         2000   
4  Recurso        2R22        18000        18000        18000        18000   

   disp_hora_5  disp_hora_6  disp_hora_7  disp_hora_8  ...  disp_hora_16  \
0        18420        18420        18420        18420  ...         18420   
1        35000        35000        35000        35000  ...         35000   
2          650          650          650          650  ...           650   
3         2000         2000         2000            0  ...          2000   
4        18000        18000        18000        18000  ...         18000   

   disp_hora_17  disp_hora_18  disp_hora_19  disp_hora_20  disp_hora_21  \

In [ ]:
# 2. Limpiar espacios en blanco y asegurar que sean texto en los códigos
df_oferta['Values_code'] = df_oferta['Values_code'].astype(str).str.strip()
df_disponibilidad['Values_code'] = df_disponibilidad['Values_code'].astype(str).str.strip()

df_oferta['Id'] = df_oferta['Id'].astype(str).str.strip()
df_disponibilidad['Id'] = df_disponibilidad['Id'].astype(str).str.strip()

# 3. Intentar el merge nuevamente
df_oferta_disp = pd.merge(
    left=df_disponibilidad, 
    right=df_oferta, 
    on=['Id','Date','Values_code'], 
    how='inner'
)

print("Filas después del merge:", len(df_oferta_disp))

Filas después del merge: 12289


In [ ]:
df_oferta_disp["Nombre_Planta"] = df_oferta["Values_code"].astype(str).str.strip().map(agentes_nombres)
print(df_oferta_disp)

# Insertar actividad

df_oferta_disp["Tecnologia"] = df_variables["listRecursos"]["Values_Name"].astype(str).str.strip().map(agentes_actividad)

            Id Values_code  disp_hora_1  disp_hora_2  disp_hora_3  \
0      Recurso        2QEK        35000        35000        35000   
1      Recurso        3DDT            0            0            0   
2      Recurso        3ENA       200000       200000       200000   
3      Recurso        3HF5            0            0            0   
4      Recurso        3INX            0            0            0   
...        ...         ...          ...          ...          ...   
12284  Recurso        URA1       338000       338000       338000   
12285  Recurso        ZPA2        34000        34000        34000   
12286  Recurso        ZPA3        63000        63000        63000   
12287  Recurso        ZPA4            0            0            0   
12288  Recurso        ZPA5        63000        63000        63000   

       disp_hora_4  disp_hora_5  disp_hora_6  disp_hora_7  disp_hora_8  ...  \
0            35000        35000        35000        35000        35000  ...   
1            

In [ ]:
display(
    HTML(
    """
    <h1>Para la construcción de la curva de oferta agregada</h1>
    <img src = "diapositiva_oferta.png">Definición de D y P para la oferta de generación.</img>
    <img src = "diapositiva3.png">Mérito de despacho</img>
    """)
)

In [ ]:
# 1. Identificar las 24 columnas de horas que quieres apilar
horas_cols = [f"oferta_hora_{x}" for x in range(1, 25)]

# 2. Identificar automáticamente todas las demás columnas (Id, Values_code, Date, Tecnología, etc.)
# Esto evita que tengas que escribirlas una por una.
otras_columnas = [col for col in df_oferta.columns if col not in horas_cols]
horas_cols_disp = [f"disp_hora_{x}" for x in range(1, 25)]
otras_columnas_disp = [col for col in df_disponibilidad.columns if col not in horas_cols_disp]
# 3. Aplicar melt para pasar las 24 columnas a filas
df_largo = df_oferta.melt(
    id_vars=otras_columnas,        # Columnas que se mantendrán fijas y cuyos valores se repetirán
    value_vars=horas_cols,         # Las 24 columnas horarias que queremos "apilar"
    var_name='Hora_Original',      # Nombre temporal para la columna que indicará la hora (ej. 'oferta_hora_1')
    value_name='Precio_Oferta'     # El valor numérico del precio para esa hora
)

df_largo_disp = df_disponibilidad.melt(
    id_vars=otras_columnas_disp,        # Columnas que se mantendrán fijas y cuyos valores se repetirán
    value_vars=horas_cols_disp,         # Las 24 columnas horarias que queremos "apilar"
    var_name='Hora_Original',      # Nombre temporal para la columna que indicará la hora (ej. 'oferta_hora_1')
    value_name='disponibilidad'     # El valor numérico del precio para esa hora
)

df_largo['Hora_Num'] = df_largo['Hora_Original'].str.extract(r'(\d+)').astype(int)
df_largo_disp['Hora_Num'] = df_largo_disp['Hora_Original'].str.extract(r'(\d+)').astype(int)

df_largo['Date'] = pd.to_datetime(df_largo['Date']) + pd.to_timedelta(df_largo['Hora_Num'] - 1, unit='h')
df_largo_disp['Date'] = pd.to_datetime(df_largo_disp['Date']) + pd.to_timedelta(df_largo_disp['Hora_Num'] - 1, unit='h')
df_largo = df_largo.drop(columns=['Hora_Original', 'Hora_Num'])
df_largo_disp = df_largo_disp.drop(columns=['Hora_Original', 'Hora_Num'])

print(df_largo_disp.head())

        Id Values_code       Date  disponibilidad
0  Recurso        2QBW 2025-10-01           18420
1  Recurso        2QEK 2025-10-01           35000
2  Recurso        2QRL 2025-10-01             650
3  Recurso        2QV2 2025-10-01            2000
4  Recurso        2R22 2025-10-01           18000


In [ ]:
df_largo_oferta_disp = pd.merge(left = df_largo, right = df_largo_disp, on = ["Id","Values_code","Date"], how = "inner")
df_largo_oferta_disp["Nombre_Planta"] = df_oferta["Values_code"].astype(str).str.strip().map(agentes_nombres)
df_largo_oferta_disp["Tecnologia"] = df_variables["listRecursos"]["Values_Name"].astype(str).str.strip().map(agentes_actividad)

print(df_largo_oferta_disp.head())

        Id Values_code       Date  Precio_Oferta  disponibilidad  \
0  Recurso        2QEK 2025-10-01         97.815           35000   
1  Recurso        3DDT 2025-10-01         88.362               0   
2  Recurso        3ENA 2025-10-01        814.351          200000   
3  Recurso        3HF5 2025-10-01         88.365               0   
4  Recurso        3INX 2025-10-01         88.000               0   

         Nombre_Planta  Tecnologia  
0             SALTO II  HIDRAULICA  
1  LATAM SOLAR LA LOMA  HIDRAULICA  
2             TESORITO  HIDRAULICA  
3            FUNDACION  HIDRAULICA  
4           CARACOLI I  HIDRAULICA  


In [ ]:
# Usamos .pivot() indicando todas las columnas que hacen única a cada fila
df_reorganizado = df_largo_oferta_disp.pivot_table(
    index=['Id', 'Values_code', 'Date', 'Precio_Oferta', 'Nombre_Planta'],
    columns='Tecnologia',
    values='disponibilidad',
    aggfunc='first'  # Toma el primer valor disponible sin modificar los números
).reset_index()

print(df_reorganizado.head())

Tecnologia       Id Values_code       Date  Precio_Oferta Nombre_Planta  \
0           Recurso        2QEK 2025-10-01         97.815      SALTO II   
1           Recurso        2QEK 2025-10-02         97.515      SALTO II   
2           Recurso        2QEK 2025-10-03         97.815      SALTO II   
3           Recurso        2QEK 2025-10-04         97.815      SALTO II   
4           Recurso        2QEK 2025-10-05         97.815      SALTO II   

Tecnologia  COGENERADOR   EOLICA  HIDRAULICA    SOLAR  Solar  TERMICA  
0                   NaN      NaN     35000.0      NaN    NaN      NaN  
1                   NaN      NaN         NaN  35000.0    NaN      NaN  
2                   NaN  35000.0         NaN      NaN    NaN      NaN  
3                   NaN      NaN         NaN  35000.0    NaN      NaN  
4                   NaN      NaN         NaN  35000.0    NaN      NaN  


In [ ]:
display(HTML(
    """
    <h1 style = "font-family: Times New Roman;">Búsqueda de variable de demanda pronósticada</h1>
    """
))

In [ ]:
df_metricas = api.get_collections()

# Puedes filtrar para buscar las palabras clave como "Demanda" o "Dema"
demandas_disponibles = df_metricas[df_metricas['MetricName'].str.contains('Demanda|Dema', case=False, na=False)]
print(demandas_disponibles)

             MetricId                                         MetricName  \
0            DemaReal                           Demanda Real por Sistema   
1            DemaReal                            Demanda Real por Agente   
3            DemaCome                      Demanda Comercial por Sistema   
12           DemaCome                       Demanda Comercial por Agente   
33             DemaOR                          Demanda por OR por Agente   
51        DemaComeReg      Demanda Comercial Mercado Regulado por Agente   
52      DemaComeNoReg   Demanda Comercial Mercado No Regulado por Agente   
53        DemaComeReg     Demanda Comercial Mercado Regulado por Sistema   
54      DemaComeNoReg  Demanda Comercial Mercado No Regulado por Sistema   
56      DemaComeNoReg                         Demanda Comercial por CIIU   
84           DemaCome      Demanda Comercial por MercadoComercializacion   
103       DemaRealReg                       Demanda Real Regulada Agente   
104       De

In [ ]:
api.get_collections("EscDemUPMEBajo")

,MetricId,MetricName,Entity,MaxDays,Type,Url,Filter,MetricUnits,MetricDescription
184,EscDemUPMEBajo,Demanda de Energia Proyectada UPME Escenario B...,Sistema,731,MonthlyEntities,https://servapibi.xm.com.co/monthly,No aplica,kWh,proyecciones de demanda de energía realizada p...


In [ ]:
df_demanda_UPME_Bajo = api.request_data("EscDemUPMEBajo","Sistema",FECHA_INICIO,FECHA_FIN)
print(df_demanda_UPME_Bajo)

        Id         Value       Date
0  Sistema  6.864000e+09 2025-10-01
1  Sistema  6.707000e+09 2025-11-01
2  Sistema  6.707000e+09 2025-11-01
3  Sistema  6.877000e+09 2025-12-01


In [ ]:
display(HTML(
"""
<h1 style = "font-family: Times New Roman;">Gráfica</h1>
<p>Para el día 1 de octubre de 2025</p>
"""
))

In [ ]:
def crear_curva_merito(df_reorganizado, df_variables, fecha_objetivo, hora=1):

    """
    Crea y muestra la curva de mérito de despacho interactiva para una fecha y hora específica.
    
    Parámetros:
    - df_reorganizado: DataFrame con la estructura de oferta y disponibilidad por tecnología.
    - df_variables: Diccionario que contiene los DataFrames con 'demanda_comercial', 'pbn', etc.
    - fecha_objetivo: String con la fecha en formato 'YYYY-MM-DD' (ej. '2025-10-01').
    - hora: Entero del 1 al 24 que indica la hora a consultar.
    """
    # 1. Copia y unificación de tecnologías si existen duplicados de mayúsculas/minúsculas
    df_temp = df_reorganizado.copy()
    if 'SOLAR' in df_temp.columns and 'Solar' in df_temp.columns:
        df_temp['SOLAR'] = df_temp['SOLAR'].fillna(df_temp['Solar'])
        df_temp = df_temp.drop(columns=['Solar'])

    cols_base = ['Id', 'Values_code', 'Date', 'Precio_Oferta', 'Nombre_Planta']
    cols_tech = [col for col in df_temp.columns if col not in cols_base]

    # Consolidar disponibilidad máxima por fila
    df_temp['disponibilidad'] = df_temp[cols_tech].max(axis=1)

    def obtener_tecnologia(row):
        for col in cols_tech:
            if pd.notna(row[col]) and row[col] > 0:
                return col
        return 'OTRA'

    df_temp['Tecnologia'] = df_temp.apply(obtener_tecnologia, axis=1)

    # 2. Filtrar para la fecha objetivo
    df_curva = df_temp[df_temp['Date'] == fecha_objetivo].copy()
    df_curva = df_curva.dropna(subset=['Precio_Oferta', 'disponibilidad'])
    df_curva = df_curva.sort_values(by='Precio_Oferta', ascending=True)

    # 3. Calcular la posición y el ancho exacto de cada rectángulo en el Eje X
    df_curva['Dispo_Acum_Fin'] = df_curva['disponibilidad'].cumsum()
    df_curva['Dispo_Acum_Inicio'] = df_curva['Dispo_Acum_Fin'] - df_curva['disponibilidad']
    df_curva['Centro_X'] = df_curva['Dispo_Acum_Inicio'] + (df_curva['disponibilidad'] / 2)

    # 4. Obtener dinámicamente la columna de la hora seleccionada (ej. 'Values_Hour01')
    col_hora = f"Values_Hour{hora:02d}"
    
    try:
        demanda_dia = float(df_variables["demanda_comercial"][df_variables["demanda_comercial"]["Date"] == fecha_objetivo][col_hora].values[0])
    except (KeyError, IndexError):
        raise ValueError(f"No se encontró la demanda comercial para la fecha {fecha_objetivo} en la hora {hora}.")

    # 5. Construcción de la figura interactiva
    fig = go.Figure()

    # Trazado de los bloques de la curva de mérito
    fig.add_trace(go.Bar(
        x=df_curva['Centro_X'],
        y=df_curva['Precio_Oferta'],
        width=df_curva['disponibilidad'], 
        customdata=df_curva[['Nombre_Planta', 'Tecnologia', 'Values_code', 'disponibilidad']],
        hovertemplate=(
            "<b>Planta:</b> %{customdata[0]}<br>"
            "<b>Tecnología:</b> %{customdata[1]}<br>"
            "<b>Código:</b> %{customdata[2]}<br>"
            "<b>Precio de Oferta:</b> %{y:,.2f} COP/kWh<br>"
            "<b>Disponibilidad:</b> %{customdata[3]:,.2f} kWh<br>"
            "<extra></extra>"
        ),
        marker=dict(
            color=pd.factorize(df_curva['Tecnologia'])[0], 
            colorscale='Viridis',
            line=dict(color='black', width=0.5) 
        )
    ))

    # Línea vertical de la demanda para esa hora
    fig.add_vline(
        x=demanda_dia,           
        line_dash="dash",             
        line_color="red",             
        line_width=2,                 
        annotation_text=f"Demanda (Hora {hora:02d})",    
        annotation_position="top right" 
    )

    # 6. Encontrar la planta marginal en la intersección
    df_marginal = df_curva[df_curva['Dispo_Acum_Fin'] >= demanda_dia]

    if not df_marginal.empty:
        planta_marginal = df_marginal.iloc[0]
        precio_marginal = planta_marginal['Precio_Oferta']
        nombre_marginal = planta_marginal['Nombre_Planta']
        tecnologia_marginal = planta_marginal['Tecnologia']
    else:
        planta_marginal = df_curva.iloc[-1]
        precio_marginal = planta_marginal['Precio_Oferta']
        nombre_marginal = planta_marginal['Nombre_Planta']
        tecnologia_marginal = planta_marginal['Tecnologia']

    # Marcador limpio en el punto exacto de intersección
    fig.add_trace(go.Scatter(
        x=[demanda_dia],
        y=[precio_marginal],
        mode='markers',
        marker=dict(size=12, color='red', symbol='x'),
        name="Planta Marginal"
    ))

    # Anotación flotante superior con planta y tecnología
    fig.add_annotation(
        x=demanda_dia,
        y=precio_marginal,
        xref="x",
        yref="y",
        text=f"<b>{nombre_marginal}</b><br>({tecnologia_marginal})<br>${precio_marginal:,.2f} COP/kWh",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="red",
        ax=0,
        ay=-140,              # Desplazamiento vertical hacia arriba
        bgcolor="white",      
        bordercolor="red",    
        borderwidth=1,
        borderpad=6
    )

    # Línea horizontal guía con el texto de corte
    fig.add_hline(
        y=precio_marginal,
        line_dash="dot",
        line_color="green"        
    )

    # 7. Ajustes finales de diseño y spikelast
    fig.update_layout(
        title=f'<b>Curva de Mérito de Despacho</b><br><sup>Fecha: {fecha_objetivo} | Hora: {hora:02d}:00</sup>',
        xaxis_title='<b>Disponibilidad Acumulada del Sistema (kWh)</b>',
        yaxis_title='<b>Precio de Oferta (COP/kWh)</b>',
        template='plotly_white',
        hovermode='x',
        bargap=0,
        xaxis=dict(
            showspikes=True,          
            spikemode='across',       
            spikesnap='cursor',       
            spikecolor='rgba(0, 0, 0, 0.5)', 
            spikethickness=1.5,       
            spikedash='dash'          
        )
    )

    fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def crear_grid_curvas_merito(df_reorganizado, df_variables, hora=1):
    """
    Crea una malla de subplots de 10x3 (hasta 30 gráficos) con la curva de mérito 
    para todos los días posibles, comenzando una semana después de la primera 
    fecha reconocida. Muestra las etiquetas del eje Y únicamente en la primera columna,
    la línea de demanda rezagada (-7 días), el precio de corte horizontal y una anotación 
    con la planta marginal, tecnología, precio de oferta y el precio de bolsa nacional (PBN) real.
    
    Parámetros:
    - df_reorganizado: DataFrame con la estructura de oferta y disponibilidad por tecnología.
    - df_variables: Diccionario que contiene los DataFrames de variables ('demanda_comercial', 'pbn', etc.).
    - hora: Entero del 1 al 24 que indica la hora a consultar en todos los subplots.
    """
    # 1. Procesar y unificar tecnologías en df_reorganizado
    df_temp = df_reorganizado.copy()
    if 'SOLAR' in df_temp.columns and 'Solar' in df_temp.columns:
        df_temp['SOLAR'] = df_temp['SOLAR'].fillna(df_temp['Solar'])
        df_temp = df_temp.drop(columns=['Solar'])

    cols_base = ['Id', 'Values_code', 'Date', 'Precio_Oferta', 'Nombre_Planta']
    cols_tech = [col for col in df_temp.columns if col not in cols_base]
    df_temp['disponibilidad'] = df_temp[cols_tech].max(axis=1)

    def obtener_tecnologia(row):
        for col in cols_tech:
            if pd.notna(row[col]) and row[col] > 0:
                return col
        return 'OTRA'

    df_temp['Tecnologia'] = df_temp.apply(obtener_tecnologia, axis=1)

    # 2. Extraer fechas únicas y calcular el punto de inicio (+7 días por la demanda rezagada)
    fechas_unicas = sorted(df_temp['Date'].unique())
    if not fechas_unicas:
        raise ValueError("No se encontraron fechas en el DataFrame de oferta.")
    
    dt_primera = pd.to_datetime(fechas_unicas[0])
    dt_inicio_plot = dt_primera + pd.Timedelta(days=7)

    fechas_objetivo = [f for f in fechas_unicas if pd.to_datetime(f) >= dt_inicio_plot]
    fechas_objetivo = fechas_objetivo[:30] # Limitado a máximo 30 elementos para la malla 10x3
    
    if not fechas_objetivo:
        raise ValueError("No hay suficientes fechas posteriores a una semana para graficar.")

    titulos = [f"Oferta: {fecha}" for fecha in fechas_objetivo]

    # 3. Configurar la malla de subplots de 10 filas x 3 columnas
    fig = make_subplots(
        rows=10, 
        cols=3,
        subplot_titles=titulos,
        vertical_spacing=0.04,
        horizontal_spacing=0.03
    )

    col_hora = f"Values_Hour{hora:02d}"

    # 4. Iterar sobre cada fecha para poblar su respectivo subplot (r, c)
    for idx, fecha_objetivo in enumerate(fechas_objetivo):
        r = (idx // 3) + 1  # Fila (1 a 10)
        c = (idx % 3) + 1   # Columna (1 a 3)

        # Calcular la fecha de la demanda (exactamente una semana antes)
        dt_obj = pd.to_datetime(fecha_objetivo)
        dt_sem_pasada = dt_obj - pd.Timedelta(days=7)
        fecha_demanda = dt_sem_pasada.strftime('%Y-%m-%d')

        # Filtrar oferta para el día objetivo actual
        df_curva = df_temp[df_temp['Date'] == fecha_objetivo].copy()
        df_curva = df_curva.dropna(subset=['Precio_Oferta', 'disponibilidad'])
        df_curva = df_curva.sort_values(by='Precio_Oferta', ascending=True)

        if df_curva.empty:
            continue

        # Calcular anchos y acumulados en el Eje X
        df_curva['Dispo_Acum_Fin'] = df_curva['disponibilidad'].cumsum()
        df_curva['Dispo_Acum_Inicio'] = df_curva['Dispo_Acum_Fin'] - df_curva['disponibilidad']
        df_curva['Centro_X'] = df_curva['Dispo_Acum_Inicio'] + (df_curva['disponibilidad'] / 2)

        # Obtener la demanda comercial correspondiente a la semana pasada
        try:
            demanda_dia = float(
                df_variables["demanda_comercial"][
                    df_variables["demanda_comercial"]["Date"] == fecha_demanda
                ][col_hora].values[0]
            )
        except (KeyError, IndexError, Exception):
            demanda_dia = 0

        # Obtener el precio de bolsa nacional real (PBN) para la fecha objetivo
        try:
            pbn_val = float(
                df_variables["pbn"][
                    df_variables["pbn"]["Date"] == fecha_objetivo
                ][col_hora].values[0]
            )
        except (KeyError, IndexError, Exception):
            pbn_val = 0.0

        # Añadir las barras de la curva de mérito al subplot (r, c)
        fig.add_trace(
            go.Bar(
                x=df_curva['Centro_X'],
                y=df_curva['Precio_Oferta'],
                width=df_curva['disponibilidad'],
                customdata=df_curva[['Nombre_Planta', 'Tecnologia', 'disponibilidad']],
                hovertemplate=(
                    "<b>Planta:</b> %{customdata[0]}<br>"
                    "<b>Tecnología:</b> %{customdata[1]}<br>"
                    "<b>Precio:</b> %{y:,.2f} COP<br>"
                    "<b>Dispo:</b> %{customdata[2]:,.2f} kWh<extra></extra>"
                ),
                marker=dict(color=pd.factorize(df_curva['Tecnologia'])[0], colorscale='Viridis'),
                showlegend=False
            ),
            row=r, col=c
        )

        if demanda_dia > 0:
            # Línea vertical de la demanda (referenciada de la semana pasada)
            fig.add_vline(
                x=demanda_dia,
                line_dash="dash",
                line_color="red",
                line_width=1,
                row=r, col=c
            )

            # Encontrar planta marginal y su precio de intersección
            df_marginal = df_curva[df_curva['Dispo_Acum_Fin'] >= demanda_dia]
            if not df_marginal.empty:
                planta_marginal = df_marginal.iloc[0]
                precio_marginal = planta_marginal['Precio_Oferta']
                nombre_marginal = planta_marginal['Nombre_Planta']
                tecnologia_marginal = planta_marginal['Tecnologia']
            else:
                planta_marginal = df_curva.iloc[-1]
                precio_marginal = planta_marginal['Precio_Oferta']
                nombre_marginal = planta_marginal['Nombre_Planta']
                tecnologia_marginal = planta_marginal['Tecnologia']

            # Marcador de intersección ('x' roja)
            fig.add_trace(
                go.Scatter(
                    x=[demanda_dia],
                    y=[precio_marginal],
                    mode='markers',
                    marker=dict(size=6, color='red', symbol='x'),
                    showlegend=False
                ),
                row=r, col=c
            )

            # Línea horizontal guía (punteada verde) para el precio de corte
            fig.add_hline(
                y=precio_marginal,
                line_dash="dot",
                line_color="green",
                line_width=1,
                row=r, col=c
            )

            # Anotación flotante con planta, tecnología, precio marginal y PBN real
            fig.add_annotation(
                x=demanda_dia,
                y=precio_marginal,
                row=r, col=c,
                text=f"<b>{nombre_marginal}</b><br>({tecnologia_marginal})<br>Oferta: ${precio_marginal:,.2f}<br>Bolsa (PBN): ${pbn_val:,.2f}",
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=1,
                arrowcolor="red",
                ax=0,
                ay=-55,               # Desplazamiento vertical adaptado para subplots
                bgcolor="white",      
                bordercolor="red",    
                borderwidth=1,
                borderpad=3,
                font=dict(size=7.5)   # Tamaño de fuente compacto para la malla
            )

    # 5. Configuración de ejes Y: Mostrar título y números solo en la columna 1, ocultar números en columnas 2 y 3
    fig.update_yaxes(title_text="<b>Precio de Oferta (COP/kWh)</b>", col=1)
    fig.update_yaxes(showticklabels=False, col=2)
    fig.update_yaxes(showticklabels=False, col=3)
    fig.update_xaxes(title_text = "<b>Disponibilidad (kWh)</b>")

    # 6. Ajustes de diseño global de la figura (malla 10x3)
    fig.update_layout(
        title=f'<b>Curvas de Mérito de Despacho - Hora {hora:02d}:00</b><br><sup>Oferta del día actual vs. Demanda de hace 7 días (Incluye Precio Bolsa Real PBN)</sup>',
        template='plotly_white',
        height=3500,   # Altura alta para acomodar las 10 filas de forma legible
        width=1400,    # Ancho para las 3 columnas
        bargap=0
    )

    fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def crear_mapa_calor_merito(df_reorganizado, df_variables, hora=1):
    """
    Crea un mapa de calor matricial de 10x3 (hasta 30 días) con la misma lógica de 
    oferta y demanda rezagada (-7 días). El color de cada celda está definido por el 
    precio marginal y contiene en su interior la información de la planta, tecnología, 
    precio de oferta y el PBN real.
    
    Parámetros:
    - df_reorganizado: DataFrame con la estructura de oferta y disponibilidad por tecnología.
    - df_variables: Diccionario que contiene los DataFrames de variables ('demanda_comercial', 'pbn', etc.).
    - hora: Entero del 1 al 24 que indica la hora a consultar en todas las celdas.
    """
    # 1. Procesar y unificar tecnologías en df_reorganizado
    df_temp = df_reorganizado.copy()
    if 'SOLAR' in df_temp.columns and 'Solar' in df_temp.columns:
        df_temp['SOLAR'] = df_temp['SOLAR'].fillna(df_temp['Solar'])
        df_temp = df_temp.drop(columns=['Solar'])

    cols_base = ['Id', 'Values_code', 'Date', 'Precio_Oferta', 'Nombre_Planta']
    cols_tech = [col for col in df_temp.columns if col not in cols_base]
    df_temp['disponibilidad'] = df_temp[cols_tech].max(axis=1)

    def obtener_tecnologia(row):
        for col in cols_tech:
            if pd.notna(row[col]) and row[col] > 0:
                return col
        return 'OTRA'

    df_temp['Tecnologia'] = df_temp.apply(obtener_tecnologia, axis=1)

    # 2. Extraer fechas únicas y calcular el punto de inicio (+7 días por la demanda rezagada)
    fechas_unicas = sorted(df_temp['Date'].unique())
    if not fechas_unicas:
        raise ValueError("No se encontraron fechas en el DataFrame de oferta.")
    
    dt_primera = pd.to_datetime(fechas_unicas[0])
    dt_inicio_plot = dt_primera + pd.Timedelta(days=7)

    fechas_objetivo = [f for f in fechas_unicas if pd.to_datetime(f) >= dt_inicio_plot]
    fechas_objetivo = fechas_objetivo[:30] # Limitado a máximo 30 elementos para la malla 10x3
    
    if not fechas_objetivo:
        raise ValueError("No hay suficientes fechas posteriores a una semana para graficar.")

    col_hora = f"Values_Hour{hora:02d}"

    # 3. Inicializar matrices de 10 filas x 3 columnas para el Heatmap
    z_matrix = [[None for _ in range(3)] for _ in range(10)]
    text_matrix = [["" for _ in range(3)] for _ in range(10)]
    hover_matrix = [["" for _ in range(3)] for _ in range(10)]
    
    y_labels = [f"Bloque {i+1}" for i in range(10)]
    x_labels = [f"Columna 1", f"Columna 2", f"Columna 3"]

    # 4. Iterar sobre cada fecha para poblar la matriz de 10x3
    for idx, fecha_objetivo in enumerate(fechas_objetivo):
        r = idx // 3  # Fila (0 a 9)
        c = idx % 3   # Columna (0 a 2)

        # Calcular la fecha de la demanda (exactamente una semana antes)
        dt_obj = pd.to_datetime(fecha_objetivo)
        dt_sem_pasada = dt_obj - pd.Timedelta(days=7)
        fecha_demanda = dt_sem_pasada.strftime('%Y-%m-%d')

        # Filtrar oferta para el día objetivo actual
        df_curva = df_temp[df_temp['Date'] == fecha_objetivo].copy()
        df_curva = df_curva.dropna(subset=['Precio_Oferta', 'disponibilidad'])
        df_curva = df_curva.sort_values(by='Precio_Oferta', ascending=True)

        if df_curva.empty:
            continue

        # Calcular disponibilidad acumulada
        df_curva['Dispo_Acum_Fin'] = df_curva['disponibilidad'].cumsum()

        # Obtener la demanda comercial de la semana pasada
        try:
            demanda_dia = float(
                df_variables["demanda_comercial"][
                    df_variables["demanda_comercial"]["Date"] == fecha_demanda
                ][col_hora].values[0]
            )
        except (KeyError, IndexError, Exception):
            demanda_dia = 0

        # Obtener el precio de bolsa nacional real (PBN) para la fecha objetivo
        try:
            pbn_val = float(
                df_variables["pbn"][
                    df_variables["pbn"]["Date"] == fecha_objetivo
                ][col_hora].values[0]
            )
        except (KeyError, IndexError, Exception):
            pbn_val = 0.0

        # Encontrar planta marginal y su precio de intersección
        if demanda_dia > 0:
            df_marginal = df_curva[df_curva['Dispo_Acum_Fin'] >= demanda_dia]
            if not df_marginal.empty:
                planta_marginal = df_marginal.iloc[0]
                precio_marginal = float(planta_marginal['Precio_Oferta'])
                nombre_marginal = str(planta_marginal['Nombre_Planta'])
                tecnologia_marginal = str(planta_marginal['Tecnologia'])
            else:
                planta_marginal = df_curva.iloc[-1]
                precio_marginal = float(planta_marginal['Precio_Oferta'])
                nombre_marginal = str(planta_marginal['Nombre_Planta'])
                tecnologia_marginal = str(planta_marginal['Tecnologia'])
        else:
            precio_marginal = 0.0
            nombre_marginal = "N/A"
            tecnologia_marginal = "N/A"

        # Asignar valores a las celdas de la matriz
        z_matrix[r][c] = precio_marginal
        # Texto incrustado dentro del cuadro de la celda
        text_matrix[r][c] = f"<b>{fecha_objetivo}</b><br>{nombre_marginal}<br>({tecnologia_marginal})<br>Oferta: ${precio_marginal:,.1f}<br>PBN: ${pbn_val:,.1f}"
        # Texto flotante detallado al pasar el cursor
        hover_matrix[r][c] = (
            f"<b>Fecha Oferta:</b> {fecha_objetivo}<br>"
            f"<b>Demanda Ref (-7d):</b> {fecha_demanda}<br>"
            f"<b>Planta Marginal:</b> {nombre_marginal}<br>"
            f"<b>Tecnología:</b> {tecnologia_marginal}<br>"
            f"<b>Precio Oferta:</b> ${precio_marginal:,.2f} COP/kWh<br>"
            f"<b>PBN Real:</b> ${pbn_val:,.2f} COP/kWh"
        )

    # 5. Construir el mapa de calor con Plotly
    fig = go.Figure(data=go.Heatmap(
        z=z_matrix,
        x=x_labels,
        y=y_labels,
        text=text_matrix,
        texttemplate="%{text}",
        hovertext=hover_matrix,
        hoverinfo="text",
        colorscale='Viridis',
        colorbar=dict(title="Precio Marginal (COP/kWh)")
    ))

    # 6. Ajustes de diseño global de la figura (dimensiones proporcionales 10x3)
    fig.update_layout(
        title=f'<b>Mapa de Calor de Curvas de Mérito - Hora {hora:02d}:00</b><br><sup>Color por Precio Marginal | Muestra Planta, Tecnología, Oferta y PBN Real</sup>',
        template='plotly_white',
        height=1200,   # Proporción ideal para 10 filas
        width=1000,    # Proporción ideal para 3 columnas
        yaxis=dict(autorange='reversed') # Invierte el eje Y para que la primera fecha inicie arriba
    )

    fig.show()

In [ ]:
crear_curva_merito(df_reorganizado,df_variables,'2025-10-01',1)

In [ ]:
crear_grid_curvas_merito(df_reorganizado, df_variables, hora=1)

In [ ]:
crear_mapa_calor_merito(df_reorganizado, df_variables, 1)